# Run All Experiments

Run-only smoke notebook for the merged Table2Charts experiment branch. It prepares the project `.venv`, uses an already processed Table2Charts corpus, then trains and evaluates the smoke versions of all experiment variants.


## 0. Project Setup


In [ ]:
from pathlib import Path
import json
import os
import re
import shlex
import subprocess
import sys
import time
from datetime import datetime, timezone

def find_project_root(start: Path = Path.cwd()) -> Path:
    markers = [".git", "README.md", "requirements.txt", "requirements-table2charts-cu118.txt", "pyproject.toml"]
    cur = start.resolve()
    for parent in [cur, *cur.parents]:
        if any((parent / marker).exists() for marker in markers):
            return parent
    return cur

PROJECT_ROOT = find_project_root()
CODE_DIR = PROJECT_ROOT / "Table2Charts"
os.chdir(PROJECT_ROOT)

DATA_ROOT = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
LOG_DIR = PROJECT_ROOT / "logs"
SUMMARY_DIR = PROJECT_ROOT / "logs" / "tensorboard"

for path in [OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR, SUMMARY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EXPERIMENT_STATE = {}

def utc_run_id() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"CODE_DIR={CODE_DIR}")


## 0.1 Environment Setup

This notebook maintains dependencies in the project-root `.venv`. The cell checks whether required packages are importable from that venv and installs only missing packages.


In [ ]:
# Ensure the project-local .venv has the packages needed by this notebook and the subprocess experiments.
import importlib.util
import venv

RUN_ENV_INSTALL = os.environ.get("T2C_RUN_ENV_INSTALL", "1").lower() not in {"0", "false", "no"}
VENV_DIR = PROJECT_ROOT / ".venv"
VENV_PYTHON = VENV_DIR / "bin" / "python"
PIP_INDEX_URL = os.environ.get("PIP_INDEX_URL", "https://mirrors.aliyun.com/pypi/simple/")
PYTORCH_WHEEL_INDEX_URL = os.environ.get("PYTORCH_WHEEL_INDEX_URL", "https://mirror.sjtu.edu.cn/pytorch-wheels/cu118")
PIP_INSTALL_TIMEOUT = os.environ.get("PIP_INSTALL_TIMEOUT", "300")
PIP_INSTALL_RETRIES = os.environ.get("PIP_INSTALL_RETRIES", "3")
PIP_NO_CACHE = os.environ.get("PIP_NO_CACHE", "0").lower() in {"1", "true", "yes"}

REQUIRED_PACKAGES = [
    ("torch", "torch==2.2.2+cu118"),
    ("torchvision", "torchvision==0.17.2+cu118"),
    ("numpy", "numpy==1.26.4"),
    ("pandas", "pandas==1.5.3"),
    ("scipy", "scipy==1.15.3"),
    ("sklearn", "scikit-learn==1.7.2"),
    ("tensorboard", "tensorboard==2.20.0"),
    ("tqdm", "tqdm==4.67.3"),
    ("sortedcontainers", "sortedcontainers==2.4.0"),
    ("pika", "pika==1.3.2"),
    ("requests", "requests==2.33.1"),
    ("PIL", "pillow==12.2.0"),
    ("ipykernel", "ipykernel>=6.29"),
]

print(f"RUN_ENV_INSTALL={RUN_ENV_INSTALL}")
print(f"VENV_DIR={VENV_DIR}")
print(f"PIP_INDEX_URL={PIP_INDEX_URL}")
print(f"PYTORCH_WHEEL_INDEX_URL={PYTORCH_WHEEL_INDEX_URL}")

if RUN_ENV_INSTALL:
    if not VENV_PYTHON.exists():
        print(f"Creating virtual environment: {VENV_DIR}")
        venv.EnvBuilder(with_pip=True, clear=False).create(VENV_DIR)

    probe = """
import importlib.util, json
mods = __import__('sys').argv[1:]
print(json.dumps({m: importlib.util.find_spec(m) is not None for m in mods}, sort_keys=True))
"""
    module_names = [module for module, _ in REQUIRED_PACKAGES]
    result = subprocess.run([str(VENV_PYTHON), "-c", probe, *module_names], text=True, capture_output=True, check=True)
    installed = json.loads(result.stdout)
    missing_specs = [spec for module, spec in REQUIRED_PACKAGES if not installed.get(module)]
    for module, spec in REQUIRED_PACKAGES:
        print(f"{'OK' if installed.get(module) else 'MISSING'} {module}: {spec}")

    if missing_specs:
        install_cmd = [
            str(VENV_PYTHON), "-m", "pip", "install",
            "-i", PIP_INDEX_URL,
            "--extra-index-url", PYTORCH_WHEEL_INDEX_URL,
            "--timeout", PIP_INSTALL_TIMEOUT,
            "--retries", PIP_INSTALL_RETRIES,
        ]
        if PIP_NO_CACHE:
            install_cmd.insert(4, "--no-cache-dir")
        install_cmd += missing_specs
        print("Installing missing packages:", " ".join(missing_specs))
        subprocess.run(install_cmd, check=True)
    else:
        print("All required packages are already available in .venv.")
else:
    print("Skipping environment setup because T2C_RUN_ENV_INSTALL=0.")


## 1. Smoke Run Config


In [ ]:
# Run-only smoke configuration. Every experiment cell executes directly in order.
RUN_MODE = os.environ.get("T2C_RUN_MODE", "full")  # "smoke" or "full"
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError('RUN_MODE must be "smoke" or "full"')

PYTHON_BIN = Path(os.environ.get("T2C_PYTHON_BIN", str(VENV_PYTHON if VENV_PYTHON.exists() else PROJECT_ROOT / ".venv" / "bin" / "python")))
CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES", "0")
CHART_TYPE = "allCharts"
MODEL_NAME = "cp"
MODEL_SIZE = "small"
FEATURES = "all-fast"
LANG = "en"
SEARCH_LIMITS = os.environ.get("T2C_SEARCH_LIMITS", "e50-b4-na")
NEGATIVE_WEIGHT = 0.8
VALID_SEARCH_LIMITS = {
    "e200-b4-r4c2", "e200-b8-r4c2", "e200-b4-na", "e200-b8-na",
    "e100-b4-r4c2", "e100-b8-r4c2", "e100-b4-na", "e100-b8-na",
    "e50-b4-r4c2", "e50-b4-na",
}
if SEARCH_LIMITS not in VALID_SEARCH_LIMITS:
    raise ValueError(f"Unsupported T2C_SEARCH_LIMITS={SEARCH_LIMITS!r}; choose one of {sorted(VALID_SEARCH_LIMITS)}")

NOTEBOOK_RUN_ROOT = PROJECT_ROOT / "notebook_runs"
NOTEBOOK_RUN_NAME = os.environ.get("T2C_NOTEBOOK_RUN_NAME", f"run_all_experiments_{RUN_MODE}_{utc_run_id()}")
NOTEBOOK_RUN_DIR = NOTEBOOK_RUN_ROOT / NOTEBOOK_RUN_NAME
OUTPUT_DIR = NOTEBOOK_RUN_DIR / "outputs"
CHECKPOINT_DIR = NOTEBOOK_RUN_DIR / "checkpoints"
LOG_DIR = NOTEBOOK_RUN_DIR / "logs"
SUMMARY_DIR = NOTEBOOK_RUN_DIR / "tensorboard"

FULL_CORPUS_PATH = DATA_ROOT / "PlotlyTable2Charts"
SMOKE_CORPUS_CANDIDATES = [
    DATA_ROOT / "PlotlyTable2ChartsSmokeSynthetic",
    DATA_ROOT / "PlotlyTable2ChartsSmokeFast",
    DATA_ROOT / "PlotlyTable2ChartsSmoke",
]
SMOKE_CORPUS_PATH = next((path for path in SMOKE_CORPUS_CANDIDATES if (path / "index" / "schema_ids.json").exists()), SMOKE_CORPUS_CANDIDATES[0])
CORPUS_PATH = Path(os.environ.get("T2C_CORPUS_PATH", str(SMOKE_CORPUS_PATH if RUN_MODE == "smoke" else FULL_CORPUS_PATH)))
SFT_CKPT = None

MODE_CONFIGS = {
    "smoke": {
        "epochs": int(os.environ.get("T2C_EPOCHS", "1")),
        "sft_nprocs": int(os.environ.get("SFT_NPROCS", "1")),
        "rl_nprocs": int(os.environ.get("RL_NPROCS", "1")),
        "eval_nprocs": int(os.environ.get("EVAL_NPROCS", "1")),
        "num_workers": int(os.environ.get("NUM_WORKERS", "0")),
        "train_batch_size": int(os.environ.get("TRAIN_BATCH_SIZE", "4")),
        "valid_batch_size": int(os.environ.get("VALID_BATCH_SIZE", "4")),
        "max_tables": int(os.environ.get("MAX_TABLES", "1")),
        "min_memory": int(os.environ.get("MIN_MEMORY", "1")),
        "memory_sample_size": int(os.environ.get("MEMORY_SAMPLE_SIZE", "1")),
        "memory_sample_rounds": int(os.environ.get("MEMORY_SAMPLE_ROUNDS", "1")),
        "eval_nagents": int(os.environ.get("EVAL_NAGENTS", "1")),
        "eval_nthreads": int(os.environ.get("EVAL_NTHREADS", "1")),
        "max_eval_tables": int(os.environ.get("MAX_EVAL_TABLES", "1")),
        "train_table_limit": int(os.environ.get("TRAIN_TABLE_LIMIT", "2")),
        "valid_table_limit": int(os.environ.get("VALID_TABLE_LIMIT", "1")),
    },
    "full": {
        "epochs": int(os.environ.get("T2C_EPOCHS", "1")),
        "sft_nprocs": int(os.environ.get("SFT_NPROCS", "1")),
        "rl_nprocs": int(os.environ.get("RL_NPROCS", "1")),
        "eval_nprocs": int(os.environ.get("EVAL_NPROCS", "1")),
        "num_workers": int(os.environ.get("NUM_WORKERS", "2")),
        "train_batch_size": int(os.environ.get("TRAIN_BATCH_SIZE", "64")),
        "valid_batch_size": int(os.environ.get("VALID_BATCH_SIZE", "64")),
        "max_tables": int(os.environ.get("MAX_TABLES", "64")),
        "min_memory": int(os.environ.get("MIN_MEMORY", "1000")),
        "memory_sample_size": int(os.environ.get("MEMORY_SAMPLE_SIZE", "64")),
        "memory_sample_rounds": int(os.environ.get("MEMORY_SAMPLE_ROUNDS", "2")),
        "eval_nagents": int(os.environ.get("EVAL_NAGENTS", "64")),
        "eval_nthreads": int(os.environ.get("EVAL_NTHREADS", "5")),
        "max_eval_tables": None,
        "train_table_limit": None,
        "valid_table_limit": None,
    },
}
CFG = MODE_CONFIGS[RUN_MODE]

FULL_REWARD_SETTINGS = {
    "conservative": {"exact": 0.95, "default": 0.05, "same_token": 0.07, "field": 0.10, "same_field_type": 0.20, "positive_threshold": 0.5},
    "current": {"exact": 0.95, "default": 0.05, "same_token": 0.10, "field": 0.15, "same_field_type": 0.35, "positive_threshold": 0.5},
    "aggressive": {"exact": 0.95, "default": 0.05, "same_token": 0.15, "field": 0.25, "same_field_type": 0.50, "positive_threshold": 0.5},
}
REWARD_SETTINGS = {"current": FULL_REWARD_SETTINGS["current"]} if RUN_MODE == "smoke" else FULL_REWARD_SETTINGS
EPSILONS = [0.05] if RUN_MODE == "smoke" else [0.05, 0.1, 0.2, 0.3]
ACTOR_EVAL_MODES = [("blend", 0.5)] if RUN_MODE == "smoke" else [("actor", 0.5), ("critic", 1.0), ("blend", 0.5)]
POLICY_EPSILON_END = 0.02
POLICY_EPSILON_DECAY = 0.8
POLICY_EXPLORE_TOP_M = 5
UCB_EXPLORATION = 0.5

for path in [NOTEBOOK_RUN_ROOT, NOTEBOOK_RUN_DIR, OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR, SUMMARY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"RUN_MODE={RUN_MODE}")
print(f"PYTHON_BIN={PYTHON_BIN}")
print(f"NOTEBOOK_RUN_DIR={NOTEBOOK_RUN_DIR}")
print(f"CORPUS_PATH={CORPUS_PATH}")
print(f"SEARCH_LIMITS={SEARCH_LIMITS}")
print(f"EPSILONS={EPSILONS}")
print(f"REWARD_SETTINGS={list(REWARD_SETTINGS)}")
print(f"ACTOR_EVAL_MODES={ACTOR_EVAL_MODES}")


## 2. Helpers


In [ ]:
def check_path(label, path, required=True):
    path = Path(path)
    ok = path.exists()
    level = "OK" if ok else ("ERROR" if required else "WARNING")
    print(f"[{level}] {label}: {path}")
    if required and not ok:
        raise FileNotFoundError(f"Missing {label}: {path}")

def as_strs(parts):
    return [str(part) for part in parts]

def quote_cmd(cmd):
    return " ".join(shlex.quote(str(part)) for part in cmd)

def experiment_paths(name: str):
    safe_name = name.replace("/", "__")
    paths = {
        "output": OUTPUT_DIR / safe_name,
        "checkpoint": CHECKPOINT_DIR / safe_name,
        "log": LOG_DIR / safe_name,
        "summary": SUMMARY_DIR / safe_name,
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths

def run_cmd(cmd, cwd=CODE_DIR, log_dir=None, env_extra=None, label="cmd"):
    cmd = as_strs(cmd)
    log_dir = Path(log_dir or LOG_DIR / "manual")
    log_dir.mkdir(parents=True, exist_ok=True)
    log_file = log_dir / f"{label}-{utc_run_id()}.log"
    env = os.environ.copy()
    env["PYTHONPATH"] = str(CODE_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    if CUDA_VISIBLE_DEVICES is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(CUDA_VISIBLE_DEVICES)
    if env_extra:
        env.update({str(k): str(v) for k, v in env_extra.items()})

    print("cwd:", cwd)
    print("log:", log_file)
    print("Running:", quote_cmd(cmd))
    result = subprocess.run(cmd, cwd=str(cwd), env=env, text=True, capture_output=True)
    log_file.write_text(
        "COMMAND: " + quote_cmd(cmd) + "\n\nSTDOUT:\n" + result.stdout + "\n\nSTDERR:\n" + result.stderr,
        encoding="utf-8",
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return result, log_file

def latest_checkpoint(checkpoint_root: Path, since: float = None) -> Path:
    candidates = list(Path(checkpoint_root).glob("**/states_ep*.pt"))
    if since is not None:
        candidates = [p for p in candidates if p.stat().st_mtime >= since]
    if not candidates:
        raise FileNotFoundError(f"No states_ep*.pt found under {checkpoint_root}")
    return max(candidates, key=lambda p: p.stat().st_mtime)

def resolve_checkpoint(experiment_key: str, checkpoint_root: Path):
    if EXPERIMENT_STATE.get(experiment_key, {}).get("checkpoint"):
        ckpt = Path(EXPERIMENT_STATE[experiment_key]["checkpoint"])
    else:
        ckpt = latest_checkpoint(checkpoint_root)
    if not ckpt.exists():
        raise FileNotFoundError(f"Checkpoint not found for {experiment_key}: {ckpt}")
    return ckpt.parent, ckpt.name, ckpt

def record_result(name: str, **kwargs):
    EXPERIMENT_STATE.setdefault(name, {})
    EXPERIMENT_STATE[name].update({k: str(v) if isinstance(v, Path) else v for k, v in kwargs.items()})
    state_path = LOG_DIR / "experiment_state.json"
    state_path.write_text(json.dumps(EXPERIMENT_STATE, indent=2, sort_keys=True), encoding="utf-8")
    return EXPERIMENT_STATE[name]

def base_train_args(checkpoint_root: Path, summary_root: Path, include_pretrained=True):
    if include_pretrained and not SFT_CKPT:
        raise FileNotFoundError("SFT_CKPT is not set. Run the SFT baseline cell before RL cells.")
    if include_pretrained and not Path(SFT_CKPT).exists():
        raise FileNotFoundError(f"SFT_CKPT does not exist: {SFT_CKPT}")
    args = [
        "--corpus_path", CORPUS_PATH,
        "--model_size", MODEL_SIZE,
        "--model_name", MODEL_NAME,
        "--features", FEATURES,
        "--negative_weight", NEGATIVE_WEIGHT,
        "--search_limits", SEARCH_LIMITS,
        "--epochs", CFG["epochs"],
        "--model_save_path", checkpoint_root,
        "--summary_path", summary_root,
        "--search_type", CHART_TYPE,
        "--input_type", CHART_TYPE,
        "--previous_type", CHART_TYPE,
        "--lang", LANG,
        "--queue_mode", "local",
        "--log_freq_agent", 500,
        "--log_freq_batch", 100,
        "--max_tables", CFG["max_tables"],
        "--min_memory", CFG["min_memory"],
        "--memory_sample_size", CFG["memory_sample_size"],
        "--memory_sample_rounds", CFG["memory_sample_rounds"],
    ]
    if include_pretrained:
        args.extend(["-p", SFT_CKPT])
    return as_strs(args)

def distributed_train(name: str, module_or_script, extra_args=None, script_mode=False, port=29640):
    paths = experiment_paths(name)
    marker = time.time()
    launcher = [PYTHON_BIN, "-m", "torch.distributed.launch", "--nproc_per_node", CFG["rl_nprocs"], "--master_port", port]
    launcher += [module_or_script] if script_mode else ["--module", module_or_script]
    cmd = launcher + base_train_args(paths["checkpoint"], paths["summary"]) + as_strs(extra_args or [])
    _, log_file = run_cmd(cmd, cwd=CODE_DIR, log_dir=paths["log"], label="train")
    ckpt = latest_checkpoint(paths["checkpoint"], since=marker)
    record_result(name, checkpoint=ckpt, train_log=log_file, checkpoint_root=paths["checkpoint"], status="trained")
    print(f"checkpoint: {ckpt}")
    return ckpt

def eval_checkpoint(name: str, checkpoint_root: Path, script="test_agent_mp.py", extra_args=None, score_mode=None, checkpoint_key=None):
    paths = experiment_paths(name)
    model_dir, model_file, ckpt = resolve_checkpoint(checkpoint_key or name, checkpoint_root)
    eval_dir = paths["output"] / "eval"
    cmd = [
        PYTHON_BIN, script,
        "-m", model_dir,
        "-f", model_file,
        "--model_name", MODEL_NAME,
        "--model_size", MODEL_SIZE,
        "--features", FEATURES,
        "--log_save_path", eval_dir,
        "--search_type", CHART_TYPE,
        "--input_type", CHART_TYPE,
        "--previous_type", CHART_TYPE,
        "--nprocs", CFG["eval_nprocs"],
        "--nagents", CFG["eval_nagents"],
        "--nthreads", CFG["eval_nthreads"],
        "--search_limits", SEARCH_LIMITS,
        "--corpus_path", CORPUS_PATH,
        "--lang", LANG,
        "--limit_search_group",
    ]
    if CFG["max_eval_tables"] is not None:
        cmd += ["--max_eval_tables", CFG["max_eval_tables"]]
    if score_mode:
        cmd += ["--score_mode", score_mode]
    cmd += as_strs(extra_args or [])
    _, log_file = run_cmd(cmd, cwd=CODE_DIR, log_dir=paths["log"], label=f"eval-{score_mode or 'test'}")
    record_result(name, checkpoint=ckpt, eval_log=log_file, eval_dir=eval_dir, status="eval_done")
    print(f"eval_dir: {eval_dir}")
    return eval_dir

def reward_args(setting: dict):
    return [
        "--update_reward_exact", setting["exact"],
        "--update_reward_default", setting["default"],
        "--update_reward_same_token", setting["same_token"],
        "--update_reward_field", setting["field"],
        "--update_reward_same_field_type", setting["same_field_type"],
        "--update_reward_positive_threshold", setting["positive_threshold"],
    ]

def policy_args(epsilon: float):
    return [
        "--policy_epsilon_start", epsilon,
        "--policy_epsilon_end", POLICY_EPSILON_END,
        "--policy_epsilon_decay", POLICY_EPSILON_DECAY,
        "--policy_explore_top_m", POLICY_EXPLORE_TOP_M,
    ]


## 3. Data Processing

The notebook defaults to `T2C_RUN_DATA_PROCESSING=auto`: it uses an existing processed Table2Charts corpus when available, otherwise it follows `dataprocess.md` by downloading `corpus.zip`, extracting `raw_data_all.csv`, and running `Data/Plotly/prepare_plotly_corpus.py`. Smoke mode caps processing by default; full mode processes the full downloaded corpus unless overridden by environment variables.


In [ ]:
# Data processing controls. The default "auto" mode keeps local processed data if present,
# and downloads/processes the public corpus only when the selected processed corpus is missing.
DATA_PROCESSING_MODE = os.environ.get("T2C_RUN_DATA_PROCESSING", "auto").lower()
if DATA_PROCESSING_MODE not in {"auto", "1", "true", "yes", "0", "false", "no"}:
    raise ValueError('T2C_RUN_DATA_PROCESSING must be "auto", 1/true/yes, or 0/false/no')

SELECTED_CORPUS_EXISTS = (CORPUS_PATH / "index" / "schema_ids.json").exists()
RUN_DATA_PROCESSING = (
    DATA_PROCESSING_MODE in {"1", "true", "yes"}
    or (DATA_PROCESSING_MODE == "auto" and not SELECTED_CORPUS_EXISTS)
)
USE_PROCESSED_CORPUS_FOR_EXPERIMENTS = RUN_DATA_PROCESSING
DATA_PROCESS_OVERWRITE = os.environ.get("T2C_DATA_PROCESS_OVERWRITE", "0").lower() in {"1", "true", "yes"}

KG4VIS_CORPUS_URL = os.environ.get("T2C_CORPUS_URL", "https://kg4vis.s3.us-east-2.amazonaws.com/corpus.zip")
CORPUS_ARCHIVE_PATH = Path(os.environ.get("T2C_CORPUS_ARCHIVE_PATH", str(PROJECT_ROOT / "corpus.zip")))
RAW_CORPUS_PATH = Path(os.environ.get("T2C_RAW_CORPUS_PATH", str(DATA_ROOT / "raw_data_all.csv")))
DATA_PROCESS_INTERMEDIATE_DIR = Path(os.environ.get("T2C_DATA_PROCESS_INTERMEDIATE_DIR", str(DATA_ROOT / "PlotlyProcessing")))
PREPARE_PLOTLY_SCRIPT = PROJECT_ROOT / "Data" / "Plotly" / "prepare_plotly_corpus.py"
DATA2VIS_CONVERTER = PROJECT_ROOT / "Data" / "Data2Vis" / "convert_to_table2charts.py"
PROCESSED_CORPUS_PATH = Path(os.environ.get(
    "T2C_PROCESSED_CORPUS_PATH",
    str(DATA_ROOT / ("PlotlyTable2ChartsSmokeGenerated" if RUN_MODE == "smoke" else "PlotlyTable2ChartsGenerated")),
))

PROCESS_LIMIT_ROWS = int(os.environ.get("T2C_PROCESS_LIMIT_ROWS", "200" if RUN_MODE == "smoke" else "0"))
PROCESS_MAX_OUTPUT_TABLES = int(os.environ.get("T2C_PROCESS_MAX_OUTPUT_TABLES", "20" if RUN_MODE == "smoke" else "0"))
PROCESS_PROGRESS_EVERY = int(os.environ.get("T2C_PROCESS_PROGRESS_EVERY", "1000" if RUN_MODE == "smoke" else "50000"))

print(f"DATA_PROCESSING_MODE={DATA_PROCESSING_MODE}")
print(f"SELECTED_CORPUS_EXISTS={SELECTED_CORPUS_EXISTS}")
print(f"RUN_DATA_PROCESSING={RUN_DATA_PROCESSING}")
print(f"USE_PROCESSED_CORPUS_FOR_EXPERIMENTS={USE_PROCESSED_CORPUS_FOR_EXPERIMENTS}")
print(f"CORPUS_PATH={CORPUS_PATH}")
print(f"PROCESSED_CORPUS_PATH={PROCESSED_CORPUS_PATH}")
print(f"KG4VIS_CORPUS_URL={KG4VIS_CORPUS_URL}")
print(f"PREPARE_PLOTLY_SCRIPT={PREPARE_PLOTLY_SCRIPT}")
print(f"DATA2VIS_CONVERTER={DATA2VIS_CONVERTER}")


In [ ]:
# Download and extract the public Plotly corpus described in dataprocess.md when processing is needed.
import zipfile
from urllib.request import urlretrieve

RAW_CORPUS_MEMBER = "raw_data_all.csv"

def download_corpus_archive():
    CORPUS_ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if CORPUS_ARCHIVE_PATH.exists() and not DATA_PROCESS_OVERWRITE:
        try:
            with zipfile.ZipFile(CORPUS_ARCHIVE_PATH) as archive:
                archive.getinfo(RAW_CORPUS_MEMBER)
            print(f"Archive already exists and looks usable: {CORPUS_ARCHIVE_PATH}")
            return CORPUS_ARCHIVE_PATH
        except Exception as exc:
            print(f"Existing archive is not usable ({exc}); downloading again.")
            CORPUS_ARCHIVE_PATH.unlink(missing_ok=True)

    print(f"Downloading {KG4VIS_CORPUS_URL} -> {CORPUS_ARCHIVE_PATH}")
    urlretrieve(KG4VIS_CORPUS_URL, CORPUS_ARCHIVE_PATH)
    return CORPUS_ARCHIVE_PATH

def extract_raw_corpus_from_archive():
    check_path("corpus archive", CORPUS_ARCHIVE_PATH)
    RAW_CORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    if RAW_CORPUS_PATH.exists() and not DATA_PROCESS_OVERWRITE:
        print(f"Raw corpus already exists: {RAW_CORPUS_PATH}")
        return RAW_CORPUS_PATH

    print(f"Extracting {RAW_CORPUS_MEMBER} from {CORPUS_ARCHIVE_PATH} -> {RAW_CORPUS_PATH}")
    with zipfile.ZipFile(CORPUS_ARCHIVE_PATH) as archive:
        if RAW_CORPUS_MEMBER not in set(archive.namelist()):
            raise FileNotFoundError(f"{RAW_CORPUS_MEMBER} was not found inside {CORPUS_ARCHIVE_PATH}")
        with archive.open(RAW_CORPUS_MEMBER) as src, RAW_CORPUS_PATH.open("wb") as dst:
            import shutil
            shutil.copyfileobj(src, dst)
    return RAW_CORPUS_PATH

if RUN_DATA_PROCESSING:
    download_corpus_archive()
    RAW_CORPUS_PATH = extract_raw_corpus_from_archive()
else:
    print("Processed corpus exists; skipping raw corpus download/extraction.")


In [ ]:
# Convert the downloaded raw corpus into Table2Charts layout when processing is needed.
check_path("prepare_plotly_corpus.py", PREPARE_PLOTLY_SCRIPT)
check_path("Data2Vis converter dependency", DATA2VIS_CONVERTER)

if RUN_DATA_PROCESSING:
    check_path("raw Plotly corpus input", RAW_CORPUS_PATH)
    DATA_PROCESS_INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        PYTHON_BIN, PREPARE_PLOTLY_SCRIPT,
        "--input", RAW_CORPUS_PATH,
        "--vizml-data-dir", DATA_PROCESS_INTERMEDIATE_DIR,
        "--output", PROCESSED_CORPUS_PATH,
        "--progress-every", PROCESS_PROGRESS_EVERY,
    ]
    if PROCESS_LIMIT_ROWS > 0:
        cmd += ["--limit-rows", PROCESS_LIMIT_ROWS]
    if PROCESS_MAX_OUTPUT_TABLES > 0:
        cmd += ["--max-output-tables", PROCESS_MAX_OUTPUT_TABLES]
    if DATA_PROCESS_OVERWRITE:
        cmd += ["--overwrite", "--overwrite-vizml-files"]
    run_cmd(cmd, cwd=PROJECT_ROOT, log_dir=LOG_DIR / "data_processing", label="prepare-plotly-corpus")

    if USE_PROCESSED_CORPUS_FOR_EXPERIMENTS:
        CORPUS_PATH = PROCESSED_CORPUS_PATH
        print(f"Using processed corpus for subsequent experiments: {CORPUS_PATH}")
else:
    print(f"Using existing processed corpus: {CORPUS_PATH}")

check_path("selected processed corpus", CORPUS_PATH)
check_path("selected corpus index", CORPUS_PATH / "index" / "schema_ids.json")


## 4. Sanity Check


In [ ]:
check_path("project root", PROJECT_ROOT)
check_path("Table2Charts code dir", CODE_DIR)
check_path("Python executable", PYTHON_BIN)
check_path("selected processed corpus", CORPUS_PATH)
check_path("corpus index", CORPUS_PATH / "index" / "schema_ids.json")
check_path("notebook run dir", NOTEBOOK_RUN_DIR)
check_path("output dir", OUTPUT_DIR)
check_path("checkpoint dir", CHECKPOINT_DIR)
check_path("log dir", LOG_DIR)

for script in [
    "nn_train/pretrain.py",
    "script.py",
    "reinforce/updated_policy_learn_dist.py",
    "reinforce/update_UCB_learn_dist.py",
    "reinforce/update_reward_learn_dist.py",
    "reinforce/update_reward_policy_learn_dist.py",
    "reinforce/update_actor_learn_dist.py",
    "reinforce/update_MC_light_learn_dist.py",
    "test_agent_mp.py",
    "update_UCB_test_agent_mp.py",
    "update_actor_test_agent_mp.py",
]:
    check_path(f"script {script}", CODE_DIR / script)

import_cmd = [PYTHON_BIN, "-c", "import torch, pandas, numpy, data, model, reinforce; print('imports_ok'); print('cuda_available', torch.cuda.is_available()); print('cuda_count', torch.cuda.device_count())"]
run_cmd(import_cmd, cwd=CODE_DIR, log_dir=LOG_DIR / "sanity", label="imports")
print(f"CUDA_VISIBLE_DEVICES={CUDA_VISIBLE_DEVICES}")


## 5. SFT Full Dataset Baseline

source: user1 code. Train with `nn_train.pretrain`; evaluate with original greedy `test_agent_mp.py`.


In [ ]:
SFT_EXP = "sft_full_baseline"
sft_paths = experiment_paths(SFT_EXP)
print(sft_paths)


In [ ]:
marker = time.time()
cmd = [
    PYTHON_BIN, "-m", "nn_train.pretrain",
    "--model_name", MODEL_NAME,
    "--model_size", MODEL_SIZE,
    "--features", FEATURES,
    "--train_batch_size", CFG["train_batch_size"],
    "--valid_batch_size", CFG["valid_batch_size"],
    "--log_freq", 200,
    "--negative_weight", NEGATIVE_WEIGHT,
    "--search_type", CHART_TYPE,
    "--input_type", CHART_TYPE,
    "--previous_type", CHART_TYPE,
    "--model_save_path", sft_paths["checkpoint"],
    "--summary_path", sft_paths["summary"],
    "--corpus_path", CORPUS_PATH,
    "--lang", LANG,
    "--epochs", CFG["epochs"],
    "--nprocs", CFG["sft_nprocs"],
    "--num_workers", CFG["num_workers"],
]
_, log_file = run_cmd(cmd, cwd=CODE_DIR, log_dir=sft_paths["log"], label="train")
sft_ckpt = latest_checkpoint(sft_paths["checkpoint"], since=marker)
SFT_CKPT = sft_ckpt
record_result(SFT_EXP, checkpoint=sft_ckpt, train_log=log_file, checkpoint_root=sft_paths["checkpoint"], status="trained")
print("SFT checkpoint:", sft_ckpt)


In [ ]:
eval_checkpoint(SFT_EXP, sft_paths["checkpoint"], script="test_agent_mp.py")


## 6. RL Full Dataset Baseline

source: user1 code. Train with `script.py` / `reinforce.learn_dist`; evaluate with original greedy `test_agent_mp.py`.


In [ ]:
RL_EXP = "rl_full_baseline"
rl_paths = experiment_paths(RL_EXP)
print(rl_paths)


In [ ]:
extra = []
if CFG["train_table_limit"] is not None:
    extra += ["--train_table_limit", CFG["train_table_limit"], "--valid_table_limit", CFG["valid_table_limit"]]
distributed_train(RL_EXP, "script.py", extra_args=extra, script_mode=True, port=29641)


In [ ]:
eval_checkpoint(RL_EXP, rl_paths["checkpoint"], script="test_agent_mp.py")


## 7. Policy Update: Epsilon-Greedy

source: user2 code. Smoke mode runs one representative epsilon; full mode runs the complete sweep.


In [ ]:
for idx, eps in enumerate(EPSILONS):
    exp = f"epsilon_greedy/eps_{eps}"
    print("\n===", exp, "===")
    distributed_train(exp, "reinforce.updated_policy_learn_dist", extra_args=policy_args(eps), port=29642 + idx)
    eval_checkpoint(exp, experiment_paths(exp)["checkpoint"], script="test_agent_mp.py")


## 8. Policy Update: UCB-Inspired Exploration

train source: user2 code (`reinforce.update_UCB_learn_dist`). test source: user1 code (`update_UCB_test_agent_mp.py`).


In [ ]:
UCB_EXP = "ucb_policy"
distributed_train(
    UCB_EXP,
    "reinforce.update_UCB_learn_dist",
    extra_args=["--ucb_exploration", UCB_EXPLORATION, "--ucb_eval_exploration"],
    port=29648,
)


In [ ]:
eval_checkpoint(
    UCB_EXP,
    experiment_paths(UCB_EXP)["checkpoint"],
    script="update_UCB_test_agent_mp.py",
    extra_args=["--ucb_exploration", UCB_EXPLORATION],
)


## 9. Reward Update + Greedy Policy

source: user2 code. Smoke mode runs the current reward setting; full mode expands the complete reward sweep.


In [ ]:
for idx, (mode, setting) in enumerate(REWARD_SETTINGS.items()):
    exp = f"reward_greedy/{mode}"
    print("\n===", exp, setting, "===")
    distributed_train(exp, "reinforce.update_reward_learn_dist", extra_args=reward_args(setting), port=29650 + idx)
    eval_checkpoint(exp, experiment_paths(exp)["checkpoint"], script="test_agent_mp.py")


## 10. Reward Update + Policy Update

source: user2 code. Defaults to current reward settings plus epsilon top-M policy update.


In [ ]:
RPU_EXP = "reward_policy_update"
distributed_train(
    RPU_EXP,
    "reinforce.update_reward_policy_learn_dist",
    extra_args=reward_args(REWARD_SETTINGS["current"]) + policy_args(0.2),
    port=29654,
)


In [ ]:
eval_checkpoint(RPU_EXP, experiment_paths(RPU_EXP)["checkpoint"], script="test_agent_mp.py")


## 11. Actor-Critic Framework

source: user2 code. Train with `reinforce.update_actor_learn_dist`; evaluate the same checkpoint with actor, critic, and blend score modes.


In [ ]:
AC_EXP = "actor_critic"
distributed_train(
    AC_EXP,
    "reinforce.update_actor_learn_dist",
    extra_args=["--actor_loss_weight", 0.1, "--entropy_weight", 0.001, "--critic_score_weight", 0.5, "--use_actor_in_eval"],
    port=29660,
)


In [ ]:
for mode, weight in ACTOR_EVAL_MODES:
    exp = f"actor_critic/{mode}_eval"
    eval_checkpoint(
        exp,
        experiment_paths(AC_EXP)["checkpoint"],
        script="update_actor_test_agent_mp.py",
        score_mode=mode,
        extra_args=["--critic_score_weight", weight],
        checkpoint_key=AC_EXP,
    )


## 12. Recommendation Diversity Experiments

Run eval-only recommendation logging for representative checkpoints, then extract chart-type diversity and unique recommendation metrics from the per-table `ranked_recommend` JSON files.


In [ ]:
import csv

RECOMMENDATION_LOG_ROOT = OUTPUT_DIR / "recommendation_diversity"
RECOMMENDATION_DIVERSITY_MANIFEST = NOTEBOOK_RUN_DIR / "recommendation_eval_manifest.csv"
RECOMMENDATION_DIVERSITY_CSV = NOTEBOOK_RUN_DIR / "recommendation_diversity.csv"
RECOMMENDATION_LOG_ROOT.mkdir(parents=True, exist_ok=True)

RECOMMENDATION_DIVERSITY_TARGETS = [
    {
        "name": "recommendation_diversity/rl_full_baseline",
        "method": "rl_full_baseline",
        "checkpoint_key": "rl_full_baseline",
        "checkpoint_root": experiment_paths("rl_full_baseline")["checkpoint"],
        "script": "test_agent_mp.py",
    },
    {
        "name": "recommendation_diversity/actor_critic_blend",
        "method": "actor_critic",
        "score_mode": "blend",
        "critic_score_weight": "0.5",
        "checkpoint_key": "actor_critic",
        "checkpoint_root": experiment_paths("actor_critic")["checkpoint"],
        "script": "update_actor_test_agent_mp.py",
        "extra_args": ["--critic_score_weight", 0.5],
    },
]

manifest_fields = [
    "run_id", "method", "reward_mode", "sampling_strategy", "epsilon_start",
    "score_mode", "critic_score_weight", "model_dir", "checkpoint", "source_log_path",
    "summary_log", "recommend_log_dir", "status", "notes",
]
manifest_rows = []

for target in RECOMMENDATION_DIVERSITY_TARGETS:
    exp = target["name"]
    print("\n===", exp, "===")
    recommend_log_dir = RECOMMENDATION_LOG_ROOT / exp.replace("/", "__")
    recommend_log_dir.mkdir(parents=True, exist_ok=True)

    extra_args = ["--recommend_log_path", recommend_log_dir] + list(target.get("extra_args", []))
    eval_dir = eval_checkpoint(
        exp,
        target["checkpoint_root"],
        script=target["script"],
        extra_args=extra_args,
        score_mode=target.get("score_mode"),
        checkpoint_key=target["checkpoint_key"],
    )

    state = EXPERIMENT_STATE[exp]
    ckpt = Path(state["checkpoint"])
    summaries = sorted(Path(eval_dir).glob("[[]test-summary]*.log"))
    manifest_rows.append({
        "run_id": NOTEBOOK_RUN_NAME,
        "method": target["method"],
        "reward_mode": "",
        "sampling_strategy": target.get("sampling_strategy", ""),
        "epsilon_start": target.get("epsilon_start", ""),
        "score_mode": target.get("score_mode", ""),
        "critic_score_weight": target.get("critic_score_weight", ""),
        "model_dir": str(ckpt.parent),
        "checkpoint": str(ckpt),
        "source_log_path": state.get("eval_log", ""),
        "summary_log": str(summaries[-1]) if summaries else "",
        "recommend_log_dir": str(recommend_log_dir),
        "status": "ok",
        "notes": "notebook recommendation diversity smoke eval",
    })

with RECOMMENDATION_DIVERSITY_MANIFEST.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=manifest_fields)
    writer.writeheader()
    writer.writerows(manifest_rows)

record_result(
    "recommendation_diversity/manifest",
    manifest=RECOMMENDATION_DIVERSITY_MANIFEST,
    recommend_log_root=RECOMMENDATION_LOG_ROOT,
    status="manifest_done",
)
print(f"Saved recommendation diversity manifest to: {RECOMMENDATION_DIVERSITY_MANIFEST}")


In [ ]:
import pandas as pd

extract_script = PROJECT_ROOT / "experiments" / "scripts" / "extract_recommendation_diversity.py"
check_path("recommendation diversity extractor", extract_script)

run_cmd(
    [
        PYTHON_BIN,
        extract_script,
        "--manifest", RECOMMENDATION_DIVERSITY_MANIFEST,
        "--output", RECOMMENDATION_DIVERSITY_CSV,
        "--overwrite",
    ],
    cwd=PROJECT_ROOT,
    log_dir=LOG_DIR / "recommendation_diversity",
    label="extract-diversity",
)

recommendation_diversity = pd.read_csv(RECOMMENDATION_DIVERSITY_CSV)
display(recommendation_diversity)
record_result(
    "recommendation_diversity/metrics",
    manifest=RECOMMENDATION_DIVERSITY_MANIFEST,
    diversity_csv=RECOMMENDATION_DIVERSITY_CSV,
    status="diversity_done",
)
print(f"Saved recommendation diversity metrics to: {RECOMMENDATION_DIVERSITY_CSV}")


## 13. Summary


In [ ]:
import pandas as pd

rows = []
for name, state in sorted(EXPERIMENT_STATE.items()):
    eval_dir = Path(state.get("eval_dir", "")) if state.get("eval_dir") else None
    summaries = sorted(eval_dir.glob("[[]test-summary]*.log")) if eval_dir and eval_dir.exists() else []
    rows.append({
        "experiment": name,
        "checkpoint": state.get("checkpoint"),
        "train_log": state.get("train_log"),
        "eval_log": state.get("eval_log"),
        "eval_dir": state.get("eval_dir"),
        "summary_log": str(summaries[-1]) if summaries else None,
        "status": "eval_done" if summaries else state.get("status", "configured"),
    })

summary = pd.DataFrame(rows)
display(summary)

metrics_csvs = [
    PROJECT_ROOT / "experiments" / "results" / "metrics.csv",
    PROJECT_ROOT / "experiments" / "results" / "final_eval_epsilon_sweep_20260425.csv",
    PROJECT_ROOT / "experiments" / "results" / "final_eval_reward_intensity_20260425.csv",
]
for csv_path in metrics_csvs:
    if csv_path.exists():
        print(f"\nExisting metrics file: {csv_path}")
        display(pd.read_csv(csv_path).head())

summary_csv = NOTEBOOK_RUN_DIR / "experiment_summary.csv"
summary.to_csv(summary_csv, index=False)
print(f"Saved notebook summary to: {summary_csv}")
